# Dual-Dataset Point-Like Analysis: Standard vs Scaled

This notebook runs a **parallel analysis** of two DL3 datasets (Standard and Scaled) over the **same set of runs**.  
Every step includes explicit debug printouts and sanity checks so problems with individual runs are caught early.

### Structure
1. Imports & configuration  
2. Load both DataStores and verify run overlap  
3. Event inspection per dataset  
4. Energy axis & geometry  
5. Dataset creation with per-run diagnostics  
6. Stacking & cumulative excess/significance checks  
7. Pivot energy & spectral model definition  
8. SED — joint fit on both datasets  
9. SED comparison plot  
10. Light Curve — run-wise fit on both datasets  
11. Light Curve comparison plot  
12. Crab-unit conversion & variability statistics  
13. Summary table

---
## 1 · Imports & configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
import astropy.units as u
import astropy.table as atbl
from astropy.coordinates import SkyCoord
from astropy.time import Time
from datetime import timedelta
import copy
from scipy.stats import chi2

from regions import PointSkyRegion, CircleSkyRegion

from gammapy.data import DataStore
from gammapy.maps import MapAxis, RegionGeom, Map
from gammapy.makers import (
    ReflectedRegionsBackgroundMaker,
    SafeMaskMaker,
    SpectrumDatasetMaker,
    WobbleRegionsFinder,
)
from gammapy.datasets import Datasets, FluxPointsDataset, SpectrumDataset
from gammapy.estimators import FluxPointsEstimator, LightCurveEstimator
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    LogParabolaSpectralModel,
    PowerLawSpectralModel,
    SkyModel,
)
from gammapy.utils.regions import compound_region_to_regions

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from gammapy import __version__ as gammapy_version
print(f"Gammapy version : {gammapy_version}")

### User configuration — edit only this cell

In [ ]:
# ── Source ──────────────────────────────────────────────────────────────────
SOURCE_NAME  = "Crab"
SOURCE_COORD = SkyCoord.from_name(SOURCE_NAME)

# ── DL3 paths ───────────────────────────────────────────────────────────────
DL3_STANDARD = "/fefs/aswg/workspace/juan.jimenez/data/real/mono/Crab/v0.12.0/Gamma/prod_standard/DL3"
DL3_SCALED   = "/fefs/aswg/workspace/juan.jimenez/data/real/mono/Crab/v0.12.0/Gamma/prod_standard/DL3scaled"

# ── Reconstructed energy range [TeV] ────────────────────────────────────────
E_RECO_MIN, E_RECO_MAX = 0.1, 20   # TeV
E_RECO_BINS = 4                    # bins per decade

# ── True energy range [TeV] (wider margins) ──────────────────────────────────
E_TRUE_MIN, E_TRUE_MAX = 0.05, 40  # TeV
E_TRUE_BINS = 6                    # bins per decade

# ── Background regions ───────────────────────────────────────────────────────
N_OFF_REGIONS = 1
# ── ON region ────────────────────────────────────────────────────────────────
THETA_CUT = 0.1   # degrees — match your DL3 cut (check RAD_MAX HDU)

# ── Safe mask ───────────────────────────────────────────────────────────────
AEFF_PERCENT = 5

# ── Light curve energy range [TeV] ──────────────────────────────────────────
E_LC_MIN, E_LC_MAX = 0.1, 20  # TeV

# ── Reference energy for initial decorrelation calculation ──────────────────
E_REF_INIT = 1.0  # TeV

# ── Labels used throughout plots ────────────────────────────────────────────
LABEL_STD    = "Standard"
LABEL_SCALED = "Scaled"
COLOR_STD    = "steelblue"
COLOR_SCALED = "tomato"

print(f"Source       : {SOURCE_NAME}")
print(f"Coordinates  : {SOURCE_COORD}")
print(f"E reco       : [{E_RECO_MIN}, {E_RECO_MAX}] TeV  ({E_RECO_BINS} bins/decade)")
print(f"E true       : [{E_TRUE_MIN}, {E_TRUE_MAX}] TeV  ({E_TRUE_BINS} bins/decade)")
print(f"N OFF regions: {N_OFF_REGIONS}")
print(f"E LC         : [{E_LC_MIN}, {E_LC_MAX}] TeV")

---
## 2 · Load both DataStores and verify run overlap

We load both directories and **check that they share exactly the same observation IDs**.  
Any mismatch is printed clearly.

In [ ]:
ds_std    = DataStore.from_dir(DL3_STANDARD)
ds_scaled = DataStore.from_dir(DL3_SCALED)

ids_std    = set(ds_std.obs_ids)
ids_scaled = set(ds_scaled.obs_ids)

ids_common      = sorted(ids_std & ids_scaled)
ids_only_std    = sorted(ids_std - ids_scaled)
ids_only_scaled = sorted(ids_scaled - ids_std)

print(f"Standard  DataStore  : {len(ids_std)} runs")
print(f"Scaled    DataStore  : {len(ids_scaled)} runs")
print(f"Common runs          : {len(ids_common)}")

if ids_only_std:
    print(f"WARNING  Only in Standard  : {ids_only_std}")
if ids_only_scaled:
    print(f"WARNING  Only in Scaled    : {ids_only_scaled}")
if not ids_only_std and not ids_only_scaled:
    print("OK  Run sets are identical — proceeding with all common runs.")

OBS_IDS = ids_common
print(f"\nObs IDs to analyse: {OBS_IDS}")

In [ ]:
obs_std    = ds_std.get_observations(OBS_IDS,    required_irf="all-optional")
obs_scaled = ds_scaled.get_observations(OBS_IDS, required_irf="all-optional")

livetime_std    = ds_std.obs_table["LIVETIME"].to(u.h).sum()
livetime_scaled = ds_scaled.obs_table["LIVETIME"].to(u.h).sum()

print(f"Standard total livetime : {livetime_std:.3f}")
print(f"Scaled   total livetime : {livetime_scaled:.3f}")
print()

# Check HDU tables — confirm RAD_MAX is present in both
for label, ds in [(LABEL_STD, ds_std), (LABEL_SCALED, ds_scaled)]:
    hdus = ds.hdu_table["HDU_TYPE"].data.tolist()
    has_rad_max = any("rad_max" in str(h).lower() for h in hdus)
    tag = "OK" if has_rad_max else "MISSING — point-like analysis will fail!"
    print(f"{label:12s}  RAD_MAX HDU: {tag}")

print("\n--- Standard HDU table ---")
display(ds_std.hdu_table)
print("\n--- Scaled HDU table ---")
display(ds_scaled.hdu_table)

In [ ]:
# Side-by-side obs tables for quick comparison
print("--- Standard obs table ---")
display(ds_std.obs_table)
print("\n--- Scaled obs table ---")
display(ds_scaled.obs_table)

---
## 3 · Event inspection per dataset

Quick look at the raw event distributions.  
Red dashed lines mark the configured `E_RECO_MIN` / `E_RECO_MAX`.

In [ ]:
%%time
energy_std_list, gammaness_std_list = [], []
energy_scaled_list, gammaness_scaled_list = [], []
for i, (obs_st, obs_sc) in enumerate(zip(obs_std, obs_scaled)):
    print(f"Reading observations data... ({i+1}/{len(OBS_IDS)})", end="\r")
    energy_std_list.append(obs_st.events.table["ENERGY"])
    gammaness_std_list.append(obs_st.events.table["GAMMANESS"])
    energy_scaled_list.append(obs_sc.events.table["ENERGY"])
    gammaness_scaled_list.append(obs_sc.events.table["GAMMANESS"])

energy_std = np.concatenate(energy_std_list)
gammaness_std = np.concatenate(gammaness_std_list)
energy_scaled = np.concatenate(energy_scaled_list)
gammaness_scaled = np.concatenate(gammaness_scaled_list)

ebins = np.logspace(-2, 1.6, 60)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), sharey=True)
for ax, energy, label, color in [
    (axes[0], energy_std,    LABEL_STD,    COLOR_STD),
    (axes[1], energy_scaled, LABEL_SCALED, COLOR_SCALED),
]:
    h = ax.hist2d(range(len(energy)), energy, (400, ebins), norm=LogNorm(), cmap="YlGnBu")
    fig.colorbar(h[3], ax=ax, label="Counts")
    ax.axhline(E_RECO_MIN, color="r", ls="--", lw=1.5, label=f"{E_RECO_MIN} TeV")
    ax.axhline(E_RECO_MAX, color="orange", ls="--", lw=1.5, label=f"{E_RECO_MAX} TeV")
    ax.set_yscale("log")
    ax.set_title(f"{label} - {len(energy):,} events")
    ax.set_xlabel("Event index (all runs stacked)")
    ax.set_ylabel("Reco Energy [TeV]")
    ax.legend(fontsize=8); ax.grid()

plt.tight_layout()
plt.show()

print(f"{'Dataset':12s}  {'N events':>10s}  {'E median [TeV]':>16s}  {'gammaness median':>18s}")
for label, en, gn in [(LABEL_STD, energy_std, gammaness_std), (LABEL_SCALED, energy_scaled, gammaness_scaled)]:
    print(f"{label:12s}  {len(en):>10,}  {np.median(en):>16.4f}  {np.median(gn):>18.4f}")

---
## 4 · Energy axes & spatial geometry

Both datasets share the **same energy binning and spatial geometry** — defined once here.

In [ ]:
energy_axis = MapAxis.from_energy_bounds(
    E_RECO_MIN, E_RECO_MAX,
    nbin=E_RECO_BINS, per_decade=True,
    unit="TeV", name="energy",
)
energy_axis_true = MapAxis.from_energy_bounds(
    E_TRUE_MIN, E_TRUE_MAX,
    nbin=E_TRUE_BINS, per_decade=True,
    unit="TeV", name="energy_true",
)

print(f"Reco energy axis edges  ({len(energy_axis.edges)-1} bins):")
display(energy_axis.edges)
print(f"\nTrue energy axis edges ({len(energy_axis_true.edges)-1} bins):")
display(energy_axis_true.edges)

In [ ]:
on_region = PointSkyRegion(center=SOURCE_COORD)
on_geom   = RegionGeom.create(region=on_region, axes=[energy_axis])

region_finder = WobbleRegionsFinder(n_off_regions=1)   # 1 is safe for any wobble geometry
maker_bkg     = ReflectedRegionsBackgroundMaker(region_finder=region_finder)

print(f"ON region  : PointSkyRegion at RA={SOURCE_COORD.ra:.4f}, Dec={SOURCE_COORD.dec:.4f}")
print(f"OFF method : WobbleRegionsFinder with 1 OFF region (RAD_MAX-driven radius)")

---
## 5 · Dataset creation with per-run diagnostics

For **every run** we print:
- Number of OFF regions found  
- Safe-mask coverage fraction  
- ON / OFF / α / excess counts per energy bin  

Runs that fail background estimation are **skipped** and listed at the end.  
Run both cells (Standard then Scaled) before moving on.

In [ ]:
maker = SpectrumDatasetMaker(
    containment_correction=False,   # must be False with point-like (RAD_MAX) IRF
    selection=["counts", "exposure", "edisp"],
)
maker_safe_mask = SafeMaskMaker(methods=["aeff-max"], aeff_percent=AEFF_PERCENT)
dataset_empty   = SpectrumDataset.create(geom=on_geom, energy_axis_true=energy_axis_true)

n_ebins = len(energy_axis.edges) - 1
print(f"Makers ready.  Safe mask: aeff-max at {AEFF_PERCENT}%   n_ebins={n_ebins}")

In [ ]:
print("="*62)
print(f"  Building STANDARD datasets  ({len(OBS_IDS)} runs)")
print("="*62)

datasets_std = Datasets()
skipped_std  = []

for obs_id, obs in zip(OBS_IDS, obs_std):
    print(f"\n--- Run {obs_id} ---")

    dataset        = maker.run(dataset_empty.copy(name=str(obs_id)), obs)
    dataset_on_off = maker_bkg.run(dataset=dataset, observation=obs)

    if dataset_on_off.counts_off is None:
        print(f"  FAILED: Background estimation returned None — run skipped.")
        skipped_std.append(obs_id)
        continue

    n_off_found    = len(compound_region_to_regions(dataset_on_off.counts_off.geom.region))
    dataset_on_off = maker_safe_mask.run(dataset_on_off, obs)
    safe_frac      = dataset_on_off.mask_safe.data.mean()

    # Skip runs where the IRF gives zero effective area at the source position
    if safe_frac == 0.0:
        print(f"  FAILED: Safe mask is all False (aeff=0 at source position) — run skipped.")
        skipped_std.append(obs_id)
        continue

    print(f"  OFF regions found : {n_off_found}")
    print(f"  Safe mask coverage: {safe_frac*100:.1f}%")
    # ... rest of the loop unchanged

    c_on   = dataset_on_off.counts.data.reshape(n_ebins)
    c_off  = dataset_on_off.counts_off.data.reshape(n_ebins)
    alpha  = dataset_on_off.alpha.data.reshape(n_ebins)
    bkg    = c_off * alpha
    excess = c_on - bkg

    print(f"  {'bin':>4s}  {'E_lo [TeV]':>10s}  {'E_hi [TeV]':>10s}  {'ON':>6s}  {'OFF':>6s}  {'alpha':>7s}  {'excess':>8s}")
    for i in range(n_ebins):
        print(f"  {i:>4d}  {energy_axis.edges[i].value:>10.4f}  {energy_axis.edges[i+1].value:>10.4f}"
              f"  {int(c_on[i]):>6d}  {int(c_off[i]):>6d}  {alpha[i]:>7.4f}  {excess[i]:>8.2f}")

    fig, ax = plt.subplots(figsize=(4.5, 2.5))
    ax.errorbar(energy_axis.center.value, c_on,  yerr=np.sqrt(c_on),  ds="steps-mid", label="ON",  marker=".")
    ax.errorbar(energy_axis.center.value, bkg,   yerr=np.sqrt(bkg),   ds="steps-mid", label="BKG", marker=".")
    ax.set_xscale("log"); ax.set_xlabel("Energy [TeV]"); ax.set_ylabel("Counts")
    ax.legend(fontsize=7); ax.grid(True, which="both", alpha=0.4)
    ax.set_title(f"{LABEL_STD}  Run {obs_id}")
    plt.tight_layout(); plt.show()

    datasets_std.append(dataset_on_off)

print("\n" + "="*62)
print(f"  Added {len(datasets_std)} / {len(OBS_IDS)} Standard runs.")
if skipped_std:
    print(f"  Skipped: {skipped_std}")
else:
    print("  No runs skipped.")
print("="*62)

In [ ]:
print("="*62)
print(f"  Building SCALED datasets  ({len(OBS_IDS)} runs)")
print("="*62)

datasets_scaled = Datasets()
skipped_scaled  = []

for obs_id, obs in zip(OBS_IDS, obs_scaled):
    print(f"\n--- Run {obs_id} ---")

    dataset        = maker.run(dataset_empty.copy(name=str(obs_id)), obs)
    dataset_on_off = maker_bkg.run(dataset=dataset, observation=obs)

    if dataset_on_off.counts_off is None:
        print(f"  FAILED: Background estimation returned None — run skipped.")
        skipped_scaled.append(obs_id)
        continue

    n_off_found    = len(compound_region_to_regions(dataset_on_off.counts_off.geom.region))
    dataset_on_off = maker_safe_mask.run(dataset_on_off, obs)
    safe_frac      = dataset_on_off.mask_safe.data.mean()

    print(f"  OFF regions found : {n_off_found}")
    print(f"  Safe mask coverage: {safe_frac*100:.1f}%")

    c_on   = dataset_on_off.counts.data.reshape(n_ebins)
    c_off  = dataset_on_off.counts_off.data.reshape(n_ebins)
    alpha  = dataset_on_off.alpha.data.reshape(n_ebins)
    bkg    = c_off * alpha
    excess = c_on - bkg

    print(f"  {'bin':>4s}  {'E_lo [TeV]':>10s}  {'E_hi [TeV]':>10s}  {'ON':>6s}  {'OFF':>6s}  {'alpha':>7s}  {'excess':>8s}")
    for i in range(n_ebins):
        print(f"  {i:>4d}  {energy_axis.edges[i].value:>10.4f}  {energy_axis.edges[i+1].value:>10.4f}"
              f"  {int(c_on[i]):>6d}  {int(c_off[i]):>6d}  {alpha[i]:>7.4f}  {excess[i]:>8.2f}")

    fig, ax = plt.subplots(figsize=(4.5, 2.5))
    ax.errorbar(energy_axis.center.value, c_on,  yerr=np.sqrt(c_on),  ds="steps-mid", label="ON",  marker=".")
    ax.errorbar(energy_axis.center.value, bkg,   yerr=np.sqrt(bkg),   ds="steps-mid", label="BKG", marker=".")
    ax.set_xscale("log"); ax.set_xlabel("Energy [TeV]"); ax.set_ylabel("Counts")
    ax.legend(fontsize=7); ax.grid(True, which="both", alpha=0.4)
    ax.set_title(f"{LABEL_SCALED}  Run {obs_id}")
    plt.tight_layout(); plt.show()

    datasets_scaled.append(dataset_on_off)

print("\n" + "="*62)
print(f"  Added {len(datasets_scaled)} / {len(OBS_IDS)} Scaled runs.")
if skipped_scaled:
    print(f"  Skipped: {skipped_scaled}")
else:
    print("  No runs skipped.")
print("="*62)

In [ ]:
# Verify both collections are symmetric before continuing
n_std    = len(datasets_std)
n_scaled = len(datasets_scaled)

print(f"Standard  datasets : {n_std}")
print(f"Scaled    datasets : {n_scaled}")

names_std    = set(d.name for d in datasets_std)
names_scaled = set(d.name for d in datasets_scaled)
only_in_std    = names_std    - names_scaled
only_in_scaled = names_scaled - names_std

if only_in_std:
    print(f"WARNING  In Standard but not Scaled : {only_in_std}")
if only_in_scaled:
    print(f"WARNING  In Scaled but not Standard : {only_in_scaled}")
if not only_in_std and not only_in_scaled:
    print("OK  Both collections contain the same runs.")

---
## 6 · Stacking & cumulative excess / significance checks

In [ ]:
stacked_std    = datasets_std.copy().stack_reduce()
stacked_scaled = datasets_scaled.copy().stack_reduce()

info_std    = datasets_std.info_table(cumulative=True)
info_scaled = datasets_scaled.info_table(cumulative=True)

print("Standard info table (cumulative):")
display(info_std)
print("\nScaled info table (cumulative):")
display(info_scaled)

In [ ]:
print("Stacked Standard dataset")
stacked_std.peek(figsize=(10, 2.5))
plt.suptitle("Stacked Standard", y=1.02)
plt.show()

print("Stacked Scaled dataset")
stacked_scaled.peek(figsize=(10, 2.5))
plt.suptitle("Stacked Scaled", y=1.02)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for info, label, color in [
    (info_std,    LABEL_STD,    COLOR_STD),
    (info_scaled, LABEL_SCALED, COLOR_SCALED),
]:
    ltime = info["livetime"].to("h")
    axes[0].plot(ltime,          info["excess"],  "o-", color=color, label=label)
    axes[1].plot(np.sqrt(ltime), info["sqrt_ts"], "o-", color=color, label=label)

axes[0].set_xlabel("Cumulative livetime [h]")
axes[0].set_ylabel("Cumulative excess")
axes[0].set_title("Excess vs Livetime")

axes[1].set_xlabel(r"$\sqrt{\mathrm{livetime}}$ [$\sqrt{\mathrm{h}}$]")
axes[1].set_ylabel(r"Significance [$\sigma$]")
axes[1].set_title(r"Significance vs $\sqrt{\mathrm{Livetime}}$")

for ax in axes:
    ax.legend(); ax.grid()

plt.tight_layout()
plt.show()

print(f"{'Dataset':12s}  {'Total excess':>14s}  {'Significance [sigma]':>22s}")
for info, label in [(info_std, LABEL_STD), (info_scaled, LABEL_SCALED)]:
    print(f"{label:12s}  {info['excess'][-1]:>14.1f}  {info['sqrt_ts'][-1]:>22.2f}")

---
## 7 · Pivot energy & spectral model

The decorrelation energy is computed from the **Standard stacked dataset** so both datasets share the same reference for direct comparison.

In [ ]:
# PWL fit on stacked Standard to extract pivot energy
_spec_pwl  = PowerLawSpectralModel(
    index=2,
    amplitude=2e-11 * u.Unit("cm-2 s-1 TeV-1"),
    reference=E_REF_INIT * u.TeV,
)
_model_pwl = SkyModel(spectral_model=_spec_pwl, name=SOURCE_NAME)
_ds_check  = stacked_std.copy(name="pivot_check")
_ds_check.models = _model_pwl

_fit_pwl = Fit()
_fit_pwl.run(datasets=_ds_check)

ref_energy = _spec_pwl.pivot_energy
print(f"Pivot (decorrelation) energy: {ref_energy.to(u.TeV):.3f}")
print("This reference energy will be used for both datasets.")

In [ ]:
# LogParabola template — deepcopy when assigning to each dataset
spectral_model_template = LogParabolaSpectralModel(reference=ref_energy)
spectral_model_template.amplitude.min = 1e-26
spectral_model_template.amplitude.max = 1e-8
spectral_model_template.alpha.min     = 0.0
spectral_model_template.beta.min      = 0.0

print("Spectral model template (LogParabola):")
display(spectral_model_template.parameters.to_table())

# Crab reference
crab_model = LogParabolaSpectralModel(
    amplitude = 3.48e-11 * u.Unit("cm-2 s-1 TeV-1"),
    reference = 1.0 * u.TeV,
    alpha     = 2.49 * u.Unit(""),
    beta      = 0.117 * u.Unit(""),
)
print("\nCrab reference (MCP Performance paper):")
display(crab_model.parameters.to_table())

---
## 8 · SED — fit on both datasets independently

In [ ]:
fpe = FluxPointsEstimator(
    energy_edges       = energy_axis.edges,
    reoptimize         = False,
    source             = SOURCE_NAME,
    selection_optional = "all",
    n_sigma_ul         = 2,
)
print(f"FluxPointsEstimator: {len(energy_axis.edges)-1} energy bins, n_sigma_ul=2")

In [ ]:
%%time
print(f"--- SED fit: {LABEL_STD} ---")

model_best_std   = SkyModel(spectral_model=spectral_model_template.copy(), name=SOURCE_NAME)
datasets_std.models = model_best_std

fit_std    = Fit()
result_std = fit_std.run(datasets=datasets_std)

print(f"  Converged  : {result_std.success}")
print(f"  Total stat : {result_std.total_stat:.3f}")
display(model_best_std.parameters.to_table())

fp_std         = fpe.run(datasets=datasets_std)
fp_dataset_std = FluxPointsDataset(data=fp_std, models=model_best_std)
print("Flux points computed.")

In [ ]:
%%time
print(f"--- SED fit: {LABEL_SCALED} ---")

model_best_scaled   = SkyModel(spectral_model=spectral_model_template.copy(), name=SOURCE_NAME)
datasets_scaled.models = model_best_scaled

fit_scaled    = Fit()
result_scaled = fit_scaled.run(datasets=datasets_scaled)

print(f"  Converged  : {result_scaled.success}")
print(f"  Total stat : {result_scaled.total_stat:.3f}")
display(model_best_scaled.parameters.to_table())

fp_scaled         = fpe.run(datasets=datasets_scaled)
fp_dataset_scaled = FluxPointsDataset(data=fp_scaled, models=model_best_scaled)
print("Flux points computed.")

In [ ]:
print(f"Flux points — {LABEL_STD}:")
display(fp_std.to_table(formatted=True, sed_type="flux"))

print(f"\nFlux points — {LABEL_SCALED}:")
display(fp_scaled.to_table(formatted=True, sed_type="flux"))

---
## 9 · SED comparison plot

In [ ]:
fig, (ax_sed, ax_res) = plt.subplots(
    2, 1, figsize=(6, 5),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

fp_dataset_std.plot_spectrum(
    ax=ax_sed,
    kwargs_fp   ={"label": LABEL_STD,    "color": COLOR_STD,    "zorder": 3},
    kwargs_model={"color": COLOR_STD,    "alpha": 0.25, "label": f"{LABEL_STD} model"},
)
fp_dataset_scaled.plot_spectrum(
    ax=ax_sed,
    kwargs_fp   ={"label": LABEL_SCALED, "color": COLOR_SCALED, "zorder": 3},
    kwargs_model={"color": COLOR_SCALED, "alpha": 0.25, "label": f"{LABEL_SCALED} model"},
)
crab_model.plot(
    ax=ax_sed,
    sed_type     ="e2dnde",
    energy_bounds=[50 * u.GeV, 30 * u.TeV],
    label        ="Crab (MCP)",
    color        ="gray",
    ls           ="--",
)

ax_sed.set_ylim(3e-12, 5e-10)
ax_sed.set_xlabel(None)
ax_sed.legend(fontsize=8)
ax_sed.grid(which="both", alpha=0.4)

fp_dataset_std.plot_residuals(ax=ax_res, method="diff/model", color=COLOR_STD)
fp_dataset_scaled.plot_residuals(ax=ax_res, method="diff/model", color=COLOR_SCALED)
ax_res.axhline(0, color="k", ls="--", lw=0.8)
ax_res.set_xlim(E_RECO_MIN * 0.8, E_RECO_MAX * 1.2)
ax_res.set_ylabel("(data-model)/model")
ax_res.grid(which="both", alpha=0.4)

plt.suptitle(f"{SOURCE_NAME}  SED comparison", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side best-fit parameter comparison
print(f"{'Parameter':20s}  {LABEL_STD:>26s}  {LABEL_SCALED:>26s}")
print("-" * 78)
for p_s, p_c in zip(model_best_std.parameters, model_best_scaled.parameters):
    if p_s.frozen:
        continue
    print(
        f"{p_s.name:20s}  {p_s.value:>13.4e} +/- {p_s.error:>8.2e}"
        f"  {p_c.value:>13.4e} +/- {p_c.error:>8.2e}"
    )

---
## 10 · Light Curve — run-wise fit on both datasets

Conventional LC approach: **PowerLaw with frozen spectral index** (only amplitude free per run).

In [ ]:
lc_spec = PowerLawSpectralModel(
    index=2.5,
    amplitude=2e-11 * u.Unit("cm-2 s-1 TeV-1"),
    reference=ref_energy,
)
lc_spec.index.frozen     = True
lc_spec.amplitude.min    = 1e-26
lc_spec.amplitude.max    = 1e-8

lc_estimator = LightCurveEstimator(
    energy_edges       = [E_LC_MIN, E_LC_MAX] * u.TeV,
    reoptimize         = False,
    selection_optional = "all",
)

print(f"LC model   : PowerLaw with index frozen to {lc_spec.index.value}")
print(f"LC energy  : [{E_LC_MIN}, {E_LC_MAX}] TeV")

In [ ]:
print(f"Running LC estimator — {LABEL_STD} ...")
datasets_std.models = SkyModel(spectral_model=lc_spec.copy(), name=SOURCE_NAME)
lc_std = lc_estimator.run(datasets_std)
lc_table_std = lc_std.to_table(sed_type="flux", format="lightcurve")
print("Done.")
display(lc_table_std)

In [ ]:
print(f"Running LC estimator — {LABEL_SCALED} ...")
datasets_scaled.models = SkyModel(spectral_model=lc_spec.copy(), name=SOURCE_NAME)
lc_scaled = lc_estimator.run(datasets_scaled)
lc_table_scaled = lc_scaled.to_table(sed_type="flux", format="lightcurve")
print("Done.")
display(lc_table_scaled)

---
## 11 · Light Curve comparison plots

In [ ]:
from datetime import timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.time import Time

fig, ax = plt.subplots(figsize=(13, 3.5))

for lc_table, label, color, offset_s in [
    (lc_table_std,    LABEL_STD,    COLOR_STD,    -20),
    (lc_table_scaled, LABEL_SCALED, COLOR_SCALED, +20),
]:
    for i, row in enumerate(lc_table):
        t_min = Time(row["time_min"], format="mjd").datetime
        t_max = Time(row["time_max"], format="mjd").datetime
        t_c   = t_min + (t_max - t_min) / 2 + timedelta(seconds=offset_s)
        t_err = (t_max - t_min) / 2

        ax.errorbar(
            t_c, row["flux"],
            yerr=[row["flux_errn"], row["flux_errp"]], 
            xerr=t_err,
            fmt="o", color=color, capsize=3, ms=4,
            label=label if i == 0 else None,
        )

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d\n%H:%M"))
ax.set_ylabel(f"Flux [{lc_std.flux.unit}]")
ax.set_xlabel("Time (UTC)")
ax.set_title(f"{SOURCE_NAME}  Light Curve  ({E_LC_MIN}–{E_LC_MAX} TeV)")
ax.legend(); ax.grid(which="both", alpha=0.4)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(lc_table_std["flux"], 20, color=COLOR_STD, histtype="step", density=True, lw=2, label="Standard")
ax.hist(lc_table_scaled["flux"], 20, color=COLOR_SCALED, histtype="step", density=True, lw=2, label="Scaled")
ax.legend()
ax.set(xlabel="Flux [1/scm2]", ylabel="Counts density", yscale="log", title="All runs")
plt.show()

In [ ]:
# Per-run significance comparison (bar chart)
n_rows   = min(len(lc_table_std), len(lc_table_scaled))
run_lbls = [str(oid) for oid in OBS_IDS[:n_rows]]
x        = np.arange(n_rows)
width    = 0.35

fig, ax = plt.subplots(figsize=(13, 3))

# FIX: Explicitly pass width, height, and color by keyword to prevent positional mixups.
# Also flat-extract the data using .flatten() just in case it's a 2D column matrix.
ax.bar(
    x=x - width/2, 
    height=np.array(lc_table_std["sqrt_ts"][:n_rows]).flatten(), 
    width=width, 
    color=COLOR_STD,    
    alpha=0.8, 
    label=LABEL_STD
)

ax.bar(
    x=x + width/2, 
    height=np.array(lc_table_scaled["sqrt_ts"][:n_rows]).flatten(), 
    width=width, 
    color=COLOR_SCALED, 
    alpha=0.8, 
    label=LABEL_SCALED
)

ax.axhline(5, color="k", ls="--", lw=0.8, label="5sigma")
ax.set_xticks(x)
ax.set_xticklabels(run_lbls, rotation=90)
ax.set_ylabel("sqrt_ts [sigma]")
ax.set_title("Per-run significance")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

---
## 12 · Crab-unit conversion & variability statistics

In [ ]:
flux_crab_integral = crab_model.integral(E_LC_MIN * u.TeV, E_LC_MAX * u.TeV)
print(f"Crab integral flux [{E_LC_MIN}–{E_LC_MAX} TeV]: {flux_crab_integral:.4e}")

n_runs_std    = len(lc_table_std)
n_runs_scaled = len(lc_table_scaled)

flux_cu_std    = lc_std.flux.data.reshape(n_runs_std)       * lc_std.flux.unit    / flux_crab_integral
flux_cu_scaled = lc_scaled.flux.data.reshape(n_runs_scaled) * lc_scaled.flux.unit / flux_crab_integral
err_cu_std     = lc_std.flux_err.data.reshape(n_runs_std)       * lc_std.flux.unit    / flux_crab_integral
err_cu_scaled  = lc_scaled.flux_err.data.reshape(n_runs_scaled) * lc_scaled.flux.unit / flux_crab_integral

n_common = min(n_runs_std, n_runs_scaled)
print(f"\n{'Run':>12s}  {LABEL_STD+' [C.U.]':>20s}  {LABEL_SCALED+' [C.U.]':>20s}")
print("-" * 60)
for i, oid in enumerate(OBS_IDS[:n_common]):
    print(f"{str(oid):>12s}  {flux_cu_std[i].value:>9.4f} +/- {err_cu_std[i].value:>8.4f}"
          f"  {flux_cu_scaled[i].value:>9.4f} +/- {err_cu_scaled[i].value:>8.4f}")

In [ ]:
run_str = np.array(OBS_IDS, dtype=str)

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.errorbar(run_str, flux_cu_std.value,    yerr=err_cu_std.value,
            fmt="o", color=COLOR_STD,    capsize=3, ms=5, label=LABEL_STD)
ax.errorbar(run_str, flux_cu_scaled.value, yerr=err_cu_scaled.value,
            fmt="s", color=COLOR_SCALED, capsize=3, ms=5, label=LABEL_SCALED)
ax.axhline(1.0, color="gray", ls="--", lw=1.2, label="1 C.U.")
ax.set_ylabel("Flux [C.U.]"); ax.set_xlabel("Run ID")
ax.set_title(f"{SOURCE_NAME}  —  Flux in Crab Units per run")
ax.legend(); ax.grid(which="both", alpha=0.4)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
for label, lc_table in [(LABEL_STD, lc_table_std), (LABEL_SCALED, lc_table_scaled)]:

    mask  = ~(np.isnan(lc_table["flux"]) | np.isnan(lc_table["flux_err"]))
    flux  = lc_table["flux"][mask]
    ferr  = lc_table["flux_err"][mask]

    w         = 1.0 / ferr**2
    mean_flux = (flux * w).sum() / w.sum()
    mean_err  = np.sqrt(1.0 / w.sum())

    chi2_val  = np.sum((flux - mean_flux)**2 / ferr**2)
    ndf       = len(flux) - 1
    p_val     = chi2.sf(chi2_val, ndf)

    flux_cu_mean = mean_flux / flux_crab_integral.value
    flux_cu_err  = mean_err  / flux_crab_integral.value

    print("="*55)
    print(f"  {label}")
    print("="*55)
    print(f"  Weighted mean flux : {mean_flux:.4e} +/- {mean_err:.2e} {lc_std.flux.unit}")
    print(f"  Mean flux [C.U.]   : {flux_cu_mean:.4f} +/- {flux_cu_err:.4f}")
    print(f"  Chi2 / ndf         : {chi2_val:.2f} / {ndf}")
    print(f"  P-value            : {p_val:.4e}")
    print()

In [ ]:
# Standard deviation of run-wise CU fluxes
print(f"Run-wise flux standard deviation [C.U.]:")
print(f"  {LABEL_STD:12s}: {np.nanstd(flux_cu_std.value):.4f}")
print(f"  {LABEL_SCALED:12s}: {np.nanstd(flux_cu_scaled.value):.4f}")

# Per-run ratio Scaled / Standard
ratio = flux_cu_scaled[:n_common].value / flux_cu_std[:n_common].value
print(f"\nPer-run flux ratio Scaled/Standard:")
for oid, r in zip(OBS_IDS[:n_common], ratio):
    arrow = "UP" if r > 1.05 else ("DOWN" if r < 0.95 else "==")
    print(f"  Run {oid}  ratio = {r:.4f}  {arrow}")

---
## 13 · Summary table

In [ ]:
n_rows = min(n_runs_std, n_runs_scaled)

t = atbl.Table()
t["run_id"]         = list(OBS_IDS[:n_rows])
t["flux_std"]       = lc_std.flux.data.reshape(n_runs_std)[:n_rows]
t["flux_std_err"]   = lc_std.flux_err.data.reshape(n_runs_std)[:n_rows]
t["flux_sca"]       = lc_scaled.flux.data.reshape(n_runs_scaled)[:n_rows]
t["flux_sca_err"]   = lc_scaled.flux_err.data.reshape(n_runs_scaled)[:n_rows]
t["flux_std_cu"]    = flux_cu_std[:n_rows].value
t["err_std_cu"]     = err_cu_std[:n_rows].value
t["flux_sca_cu"]    = flux_cu_scaled[:n_rows].value
t["err_sca_cu"]     = err_cu_scaled[:n_rows].value
t["ratio_sca_std"]  = ratio[:n_rows]
t["sqrt_ts_std"]    = list(lc_table_std["sqrt_ts"][:n_rows])
t["sqrt_ts_sca"]    = list(lc_table_scaled["sqrt_ts"][:n_rows])

# Format floats
for col in t.colnames:
    if t[col].dtype.kind == "f":
        t[col].format = ".4f"

display(t)

outfile = "summary_lc_comparison.ecsv"
t.write(outfile, format="ascii.ecsv", overwrite=True)
print(f"Saved to {outfile}")